In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import mne

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:

# Load individual EEG 
DATA_ROOT = Path("/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives")

def load_eeg(subject, session):
    
    path = os.path.join(
        ROOT,
        f"sub-{subject:02d}",
        f"ses-{session:02d}",
        f"sub-{subject:02d}_ses-{session:02d}_eeg-epo.fif"
    )

    epochs = mne.read_epochs(
        path,
        preload=True,
        verbose=False
    )

    return epochs

In [ ]:
eeg_path = list(DATA_ROOT.rglob("*_eeg-epo.fif"))[0]

epochs = mne.read_epochs(
    eeg_path,
    preload=True,
    verbose=False
)

X = epochs.get_data()

print("File:", eeg_path)
print("Shape:", X.shape)
print("Sampling rate:", epochs.info["sfreq"])
print("Channels:", len(epochs.ch_names))
print("Time range:", epochs.times[0], "to", epochs.times[-1])


event_path = Path(
    str(eeg_path).replace("_eeg-epo.fif", "_events.dat")
)

events = pd.read_pickle(event_path)



In [ ]:
X = epochs.get_data()

print("EEG shape:", X.shape)
print("Events shape:", events.shape)

assert len(X) == len(events)

print("Classes:", np.unique(events[:, 1], return_counts=True))
print("Conditions:", np.unique(events[:, 2], return_counts=True))
print("Sessions:", np.unique(events[:, 3], return_counts=True))



In [ ]:
CLASS_NAMES = {
    0: "Up",
    1: "Down",
    2: "Right",
    3: "Left"
}
# condition == 1 means Inner Speech
inner_mask = events[:, 2] == 1

X_inner = X[inner_mask]
y_inner = events[inner_mask, 1].astype(np.int64)

print("All trials:", len(X))
print("Inner-speech trials:", len(X_inner))

for label in np.unique(y_inner):
    print(
        CLASS_NAMES[label],
        np.sum(y_inner == label)
    )



In [ ]:
# Keep the signal 1.0s-3.5s. That's the official action interval.
times = epochs.times

action_mask = (times >= 1.0) & (times < 3.5)

X_inner = X_inner[:, :, action_mask]

print(X_inner.shape)

In [ ]:
import plotly.graph_objects as go

trial = 0
channel = 0

signal = X_inner[trial, channel] * 1e6  # volts -> µV

time_action = times[action_mask]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=time_action,
        y=signal,
        mode="lines",
        name=epochs.ch_names[channel]
    )
)

fig.update_layout(
    title=f"Inner Speech: {CLASS_NAMES[y_inner[trial]]}",
    xaxis_title="Time (seconds)",
    yaxis_title="Amplitude (µV)"
)

fig.show()

# Exploring the EEG channels on the area of the scalp

In [ ]:
import mne
import numpy as np
import pandas as pd


montage = mne.channels.make_standard_montage("biosemi128")

epochs.set_montage(
    montage,
    match_case=False,
    on_missing="warn"
)



ch_pos = epochs.get_montage().get_positions()["ch_pos"]

rows = []

for ch in epochs.ch_names:

    if ch not in ch_pos:
        continue

    x, y, z = ch_pos[ch]

    rows.append({
        "channel": ch,
        "x": x,   # left (-) / right (+)
        "y": y,   # back (-) / front (+)
        "z": z    # bottom / top
    })

channel_df = pd.DataFrame(rows)

print(channel_df.head())

In [ ]:
# --------------------------------------------------
# 3. Assign rough scalp regions from coordinates
# --------------------------------------------------

def get_region(x, y, z):

    # Front-back classification
    if y > 0.045:
        region = "Frontal"

    elif y > 0.015:
        region = "Front-Central"

    elif y > -0.020:
        region = "Central"

    elif y > -0.050:
        region = "Parietal"

    else:
        region = "Occipital"

    # Strong lateral position → Temporal
    if abs(x) > 0.060 and -0.045 < y < 0.045:
        region = "Temporal"

    return region


channel_df["region"] = channel_df.apply(
    lambda row: get_region(
        row["x"],
        row["y"],
        row["z"]
    ),
    axis=1
)
# --------------------------------------------------
# 4. Assign hemisphere
# --------------------------------------------------

def get_hemisphere(x, threshold=0.005):

    if x < -threshold:
        return "Left"

    elif x > threshold:
        return "Right"

    else:
        return "Midline"


channel_df["hemisphere"] = channel_df["x"].apply(
    get_hemisphere
)

In [ ]:
# --------------------------------------------------
# 5. View channel information
# --------------------------------------------------

channel_df[
    [
        "channel",
        "region",
        "hemisphere",
        "x",
        "y",
        "z"
    ]
].sort_values(
    ["region", "hemisphere"]
)

In [ ]:
# --------------------------------------------------
# 7. Get channels belonging to a specific region
# --------------------------------------------------

frontal_channels = channel_df[
    channel_df["region"] == "Frontal"
]["channel"].tolist()

temporal_channels = channel_df[
    channel_df["region"] == "Temporal"
]["channel"].tolist()

central_channels = channel_df[
    channel_df["region"] == "Central"
]["channel"].tolist()

parietal_channels = channel_df[
    channel_df["region"] == "Parietal"
]["channel"].tolist()

occipital_channels = channel_df[
    channel_df["region"] == "Occipital"
]["channel"].tolist()

print("Frontal:", frontal_channels)
print("Temporal:", temporal_channels)
print("Central:", central_channels)
print("Parietal:", parietal_channels)
print("Occipital:", occipital_channels)

In [ ]:
epochs.plot_sensors(
    kind="3d",
    show_names=True
)

In [ ]:
channel_df.to_csv(
    "biosemi128_channel_regions.csv",
    index=False
)

# Create channels configs for further experiment

In [ ]:
import numpy as np

# --------------------------------------------------
# 1. Get channel positions in the SAME order as X
# --------------------------------------------------

ch_pos = epochs.get_montage().get_positions()["ch_pos"]

channel_names = np.array(epochs.ch_names)

positions = np.array([
    ch_pos[ch]
    for ch in channel_names
])

print(channel_names.shape)
print(positions.shape)

# Expected:
# (128,)
# (128, 3)

In [ ]:
# --------------------------------------------------
# 2. Create spatially distributed channel ranking
#    using Farthest Point Sampling
# --------------------------------------------------

def farthest_point_sampling(positions):

    n_channels = len(positions)

    # Start from electrode closest to top/center of scalp
    center_target = np.array([
        0,
        0,
        positions[:, 2].max()
    ])

    start = np.argmin(
        np.linalg.norm(
            positions - center_target,
            axis=1
        )
    )

    selected = [start]

    remaining = set(
        range(n_channels)
    )

    remaining.remove(start)

    while remaining:

        best_channel = None
        best_distance = -1

        for candidate in remaining:

            distances = np.linalg.norm(
                positions[candidate]
                - positions[selected],
                axis=1
            )

            # Distance to nearest selected electrode
            min_distance = distances.min()

            if min_distance > best_distance:

                best_distance = min_distance
                best_channel = candidate

        selected.append(
            best_channel
        )

        remaining.remove(
            best_channel
        )

    return np.array(selected)
# --------------------------------------------------
# 3. Generate full spatial ranking
# --------------------------------------------------

channel_order = farthest_point_sampling(
    positions
)

print(channel_order)

In [ ]:
# --------------------------------------------------
# 4. Create nested channel configurations
# --------------------------------------------------

channel_configs_idx = {
    128: channel_order[:128],
    64:  channel_order[:64],
    32:  channel_order[:32],
    16:  channel_order[:16]
}

In [ ]:
# --------------------------------------------------
# 5. Convert indices -> channel names
# --------------------------------------------------

channel_configs = {
    n: channel_names[idx].tolist()
    for n, idx in channel_configs_idx.items()
}

for n_channels, channels in channel_configs.items():

    print(
        f"\n{n_channels} CHANNELS"
    )

    print(channels)

In [ ]:
assert set(channel_configs[16]).issubset(
    channel_configs[32]
)

assert set(channel_configs[32]).issubset(
    channel_configs[64]
)

assert set(channel_configs[64]).issubset(
    channel_configs[128]
)

print("All configurations are nested correctly.")

In [ ]:
for n_channels, channels in channel_configs.items():

    subset = channel_df[
        channel_df["channel"].isin(
            channels
        )
    ]

    print(
        f"\n===== {n_channels} channels ====="
    )

    print(
        subset.groupby(
            ["region", "hemisphere"]
        ).size()
    )

In [ ]:
import matplotlib.pyplot as plt
import mne

for n_channels, channels in channel_configs.items():

    fig, ax = plt.subplots(
        figsize=(8, 8)
    )

    picks = [
        epochs.ch_names.index(ch)
        for ch in channels
    ]

    mne.viz.plot_sensors(
        epochs.info,
        kind="topomap",
        ch_type="eeg",
        show_names=False,
        axes=ax,
        show=False
    )

    xy = mne.channels.layout._find_topomap_coords(
        epochs.info,
        picks=picks
    )

    ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=40
    )

    ax.set_title(
        f"{n_channels}-Channel Configuration"
    )

    plt.show()

In [ ]:
X_configs = {
    n: X[:, idx, :]
    for n, idx
    in channel_configs_idx.items()
}

for n, data in X_configs.items():
    print(n, data.shape)

In [ ]:
np.save(
    "channel_configs_idx.npy",
    channel_configs_idx,
    allow_pickle=True
)

np.save(
    "channel_configs_names.npy",
    channel_configs,
    allow_pickle=True
)

# Load all subjects

In [ ]:
all_X = []
all_y = []
all_subjects = []
all_sessions = []

eeg_files = sorted(
    DATA_ROOT.rglob("*_eeg-epo.fif")
)

print("Number of EEG files:", len(eeg_files))


In [ ]:
for file_number, eeg_path in enumerate(eeg_files):

    # -----------------------
    # Load EEG
    # -----------------------
    epochs = mne.read_epochs(
        eeg_path,
        preload=True,
        verbose=False
    )

    X = epochs.get_data()

    # -----------------------
    # Load events
    # -----------------------
    event_path = Path(
        str(eeg_path).replace(
            "_eeg-epo.fif",
            "_events.dat"
        )
    )

    events = pd.read_pickle(event_path)
    events = np.asarray(events)

    assert len(X) == len(events)

    # -----------------------
    # Subject and session IDs
    # -----------------------
    subject = int(
        re.search(
            r"sub-(\d+)",
            str(eeg_path)
        ).group(1)
    )
    session_match = re.search(r"ses-([^/\_]+)", str(eeg_path))
    session_token = (
        session_match.group(1)
        if session_match is not None
        else f"file-{file_number:03d}"
    )
    session_id = f"sub-{subject:02d}_ses-{session_token}"

    # -----------------------
    # Inner Speech only
    # -----------------------
    inner_mask = events[:, 2] == 1

    X = X[inner_mask]
    y = events[inner_mask, 1].astype(np.int64)

    # -----------------------
    # Official action interval
    # -----------------------
    times = epochs.times

    action_mask = (
        (times >= 1.0) &
        (times < 3.5)
    )

    X = X[:, :, action_mask]

    # -----------------------
    # Save
    # -----------------------
    all_X.append(X)
    all_y.append(y)

    all_subjects.append(
        np.full(
            len(X),
            subject,
            dtype=np.int64
        )
    )
    all_sessions.append(
        np.full(
            len(X),
            session_id,
            dtype=object
        )
    )


In [ ]:
X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
subjects = np.concatenate(all_subjects, axis=0)
sessions = np.concatenate(all_sessions, axis=0)

print("X:", X.shape)
print("y:", y.shape)
print("subjects:", subjects.shape)
print("sessions:", sessions.shape)
print("Unique sessions:", len(np.unique(sessions)))


In [ ]:
print("NaN:", np.isnan(X).sum())
print("Inf:", np.isinf(X).sum())

print("\nClasses:")
for label in range(4):
    print(
        CLASS_NAMES[label],
        np.sum(y == label)
    )

print("\nSubjects:")
for subject in np.unique(subjects):
    print(
        f"Subject {subject}:",
        np.sum(subjects == subject)
    )

# Machine learning approach

In [ ]:
# Split by subject

train_subjects = [1, 2, 3, 4, 5, 6, 7]
val_subjects   = [8]
test_subjects  = [9, 10]

train_mask = np.isin(subjects, train_subjects)
val_mask   = np.isin(subjects, val_subjects)
test_mask  = np.isin(subjects, test_subjects)

X_train = X[train_mask]
y_train = y[train_mask]

X_val = X[val_mask]
y_val = y[val_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
# Using welch method 
from scipy.signal import welch
def extract_bandpower_features(X, fs):

    bands = {
        "delta": (1, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta":  (13, 30),
        "gamma": (30, 45)
    }

    all_features = []

    for trial in X:

        # trial:
        # [channels, time]

        freqs, psd = welch(
            trial,
            fs=fs,
            nperseg=min(256, trial.shape[-1]),
            axis=-1
        )

        trial_features = []

        for low, high in bands.values():

            mask = (
                (freqs >= low) &
                (freqs < high)
            )

            band_power = psd[:, mask].mean(axis=1)

            # Log-transform power
            band_power = np.log(
                band_power + 1e-12
            )

            trial_features.append(
                band_power
            )

        # [5 bands, 128 channels]
        trial_features = np.array(
            trial_features
        )

        # Flatten:
        # 5 × 128 = 640 features
        trial_features = trial_features.flatten()

        all_features.append(
            trial_features
        )

    return np.array(all_features)

In [ ]:
fs = epochs.info["sfreq"] # Sampling rate

print(fs)

X_train_ml = extract_bandpower_features(
    X_train,
    fs
)

X_val_ml = extract_bandpower_features(
    X_val,
    fs
)

X_test_ml = extract_bandpower_features(
    X_test,
    fs
)

print(X_train_ml.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_ml = scaler.fit_transform(
    X_train_ml
)

X_val_ml = scaler.transform(
    X_val_ml
)

X_test_ml = scaler.transform(
    X_test_ml
)

In [ ]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

svm.fit(
    X_train_ml,
    y_train
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

val_pred = svm.predict(
    X_val_ml
)

print(
    "Val Accuracy:",
    accuracy_score(
        y_val,
        val_pred
    )
)

print(
    "Val Macro F1:",
    f1_score(
        y_val,
        val_pred,
        average="macro"
    )
)
test_pred = svm.predict(
    X_test_ml
)

print(
    "Test Accuracy:",
    accuracy_score(
        y_test,
        test_pred
    )
)

print(
    "Test Macro F1:",
    f1_score(
        y_test,
        test_pred,
        average="macro"
    )
)

# Deep learning baseline


In [ ]:
mean = X_train.mean(
    axis=(0, 2),
    keepdims=True
)

std = X_train.std(
    axis=(0, 2),
    keepdims=True
)

std = np.maximum(
    std,
    1e-8
)

X_train_dl = (
    X_train - mean
) / std

X_val_dl = (
    X_val - mean
) / std

X_test_dl = (
    X_test - mean
) / std

In [ ]:
print(X_train_dl.shape)

print(
    X_train_dl.mean(),
    X_train_dl.std()
)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class EEGDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx]
        )

train_dataset = EEGDataset(
    X_train_dl,
    y_train
)

val_dataset = EEGDataset(
    X_val_dl,
    y_val
)

test_dataset = EEGDataset(
    X_test_dl,
    y_test
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
# EEGNet style baseline
import torch.nn as nn


class EEGNet(nn.Module):

    def __init__(
        self,
        n_channels,
        n_classes=4,
        dropout=0.5
    ):

        super().__init__()

        F1 = 8
        D = 2
        F2 = F1 * D

        # Temporal filtering
        self.temporal = nn.Sequential(

            nn.Conv2d(
                1,
                F1,
                kernel_size=(1, 64),
                padding=(0, 32),
                bias=False
            ),

            nn.BatchNorm2d(F1)
        )

        # Spatial filtering across EEG channels
        self.spatial = nn.Sequential(

            nn.Conv2d(
                F1,
                F1 * D,
                kernel_size=(n_channels, 1),
                groups=F1,
                bias=False
            ),

            nn.BatchNorm2d(F1 * D),

            nn.ELU(),

            nn.AvgPool2d(
                kernel_size=(1, 4)
            ),

            nn.Dropout(dropout)
        )

        # Separable temporal convolution
        self.separable = nn.Sequential(

            nn.Conv2d(
                F2,
                F2,
                kernel_size=(1, 16),
                padding=(0, 8),
                groups=F2,
                bias=False
            ),

            nn.Conv2d(
                F2,
                F2,
                kernel_size=(1, 1),
                bias=False
            ),

            nn.BatchNorm2d(F2),

            nn.ELU(),

            nn.AvgPool2d(
                kernel_size=(1, 8)
            ),

            nn.Dropout(dropout)
        )

        self.pool = nn.AdaptiveAvgPool2d(
            (1, 1)
        )

        self.classifier = nn.Linear(
            F2,
            n_classes
        )

    def forward(self, x):

        # [B, channels, time]
        # ->
        # [B, 1, channels, time]

        x = x.unsqueeze(1)

        x = self.temporal(x)

        x = self.spatial(x)

        x = self.separable(x)

        x = self.pool(x)

        x = x.flatten(1)

        return self.classifier(x)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = EEGNet(
    n_channels=X_train_dl.shape[1],
    n_classes=4
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
from tqdm import tqdm

def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)

        pred = logits.argmax(dim=1)

        correct += (
            pred == y_batch
        ).sum().item()

        total += len(y_batch)

    return (
        total_loss / total,
        correct / total
    )

In [ ]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    total_loss = 0
    total = 0

    predictions = []
    labels = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)

        loss = criterion(
            logits,
            y_batch
        )

        total_loss += loss.item() * len(y_batch)
        total += len(y_batch)

        pred = logits.argmax(dim=1)

        predictions.extend(
            pred.cpu().numpy()
        )

        labels.extend(
            y_batch.cpu().numpy()
        )

    predictions = np.array(predictions)
    labels = np.array(labels)

    accuracy = (
        predictions == labels
    ).mean()

    return (
        total_loss / total,
        accuracy,
        labels,
        predictions
    )

In [ ]:
from tqdm import tqdm

EPOCHS = 50
best_val = 0

pbar = tqdm(
    range(EPOCHS),
    desc="Training EEGNet"
)

for epoch in pbar:

    train_loss, train_acc = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc, _, _ = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    if val_acc > best_val:

        best_val = val_acc

        torch.save(
            model.state_dict(),
            "best_eegnet.pt"
        )

    pbar.set_postfix(
        train_acc=f"{train_acc:.3f}",
        val_acc=f"{val_acc:.3f}",
        best=f"{best_val:.3f}",
        loss=f"{train_loss:.3f}"
    )

# Recommended three-pipeline experiments

This section implements the selected final design across the fixed **128 / 64 / 32 / 16** nested channel configurations:

1. **Paper-inspired classical model:** rich time, frequency, and wavelet features → fold-local variance filtering → approximate MRMR → fold-local standardization → tuned SVM.
2. **Improved scratch model:** EEGNet with training-only normalization, mild training-only augmentation, AdamW, cosine scheduling, early stopping, and repeated seeds.
3. **Pretrained transfer model:** LaBraM-Base with a fixed geometry-derived projection from the BioSemi montage to LaBraM's canonical 128-position montage, followed by linear probing, partial fine-tuning, and full fine-tuning.

The default development split remains subjects 1–7 / subject 8 / subjects 9–10. Test data never controls preprocessing, feature selection, hyperparameters, learning-rate schedules, or early stopping. Optional functions at the end run rigorous LOSO and paper-comparable subject-specific analyses.

**Run modes:** `quick` uses one deep-learning seed and a compact SVM grid. Change `RUN_MODE` to `final` for five seeds and the complete search. LaBraM stays opt-in because its checkpoint must be downloaded once.


In [ ]:
import copy
import hashlib
import json
import math
import random
import warnings
from fractions import Fraction

import pywt
from scipy.signal import resample_poly, welch
from scipy.stats import kurtosis, skew
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import (
    ParameterGrid,
    StratifiedGroupKFold,
    StratifiedKFold,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    import cupy as cp
    from cuml.svm import SVC as CuMLSVC

    GPU_SVM_AVAILABLE = cp.cuda.runtime.getDeviceCount() > 0
    if GPU_SVM_AVAILABLE:
        cp.cuda.Device(0).use()
except Exception as gpu_svm_error:
    cp = None
    CuMLSVC = None
    GPU_SVM_AVAILABLE = False
    warnings.warn(f"GPU SVM unavailable; using CPU: {gpu_svm_error}")


RUN_MODE = "quick"  # Change to "final" for the report-quality experiment.
if RUN_MODE not in {"quick", "final"}:
    raise ValueError("RUN_MODE must be 'quick' or 'final'.")

CHANNEL_COUNTS = (128, 64, 32, 16)
FINAL_SEEDS = (13, 42, 101, 202, 999)
ACTIVE_DEEP_SEEDS = (42,) if RUN_MODE == "quick" else FINAL_SEEDS

RUN_DEVELOPMENT_RICH_SVM = False
RUN_DEVELOPMENT_EEGNET = False
RUN_DEVELOPMENT_LABRAM = False
RUN_LABRAM_RANDOM_CONTROL = False

RESULTS_DIR = Path("results_recommended")
CHECKPOINT_DIR = Path("checkpoints_recommended")
CACHE_DIR = RESULTS_DIR / "cache"
for directory in (RESULTS_DIR, CHECKPOINT_DIR, CACHE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

DEV_SPLIT = {
    "train": (1, 2, 3, 4, 5, 6, 7),
    "val": (8,),
    "test": (9, 10),
}

EXPERIMENT_ROWS = {}
PREDICTION_FRAMES = {}


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def validate_subject_split(split):
    groups = [set(split[name]) for name in ("train", "val", "test")]
    assert all(groups), "Train, validation, and test groups must be non-empty."
    assert not (groups[0] & groups[1] or groups[0] & groups[2] or groups[1] & groups[2])
    assert set.union(*groups).issubset(set(np.unique(subjects).tolist()))


def subject_split_masks(split):
    validate_subject_split(split)
    return {
        name: np.isin(subjects, split[name])
        for name in ("train", "val", "test")
    }


def channel_config_hash(n_channels):
    payload = json.dumps(channel_configs[n_channels], separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()[:12]


def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def store_predictions(protocol, fold, model_name, n_channels, partition, mask, y_pred, seed):
    key = (protocol, fold, model_name, n_channels, partition, seed)
    PREDICTION_FRAMES[key] = pd.DataFrame({
        "protocol": protocol,
        "fold": fold,
        "model": model_name,
        "n_channels": n_channels,
        "partition": partition,
        "seed": seed,
        "subject": subjects[mask],
        "session": sessions[mask],
        "y_true": y[mask],
        "y_pred": np.asarray(y_pred),
    })


def store_result(
    protocol,
    fold,
    model_name,
    n_channels,
    split,
    val_metrics,
    test_metrics,
    seed,
    best_epoch=None,
    config_extra=None,
):
    row = {
        "protocol": protocol,
        "fold": fold,
        "model": model_name,
        "n_channels": n_channels,
        "channel_config_hash": channel_config_hash(n_channels),
        "train_subjects": ",".join(map(str, split["train"])),
        "val_subjects": ",".join(map(str, split["val"])),
        "test_subjects": ",".join(map(str, split["test"])),
        "input_sfreq": float(fs),
        "time_start_s": 1.0,
        "time_stop_s": 3.5,
        "seed": seed,
        "best_epoch": best_epoch,
        **{f"val_{key}": value for key, value in val_metrics.items()},
        **{f"test_{key}": value for key, value in test_metrics.items()},
    }
    if config_extra:
        row.update(config_extra)
    EXPERIMENT_ROWS[(protocol, fold, model_name, n_channels, seed)] = row
    return row


experiment_info = mne.create_info(
    ch_names=channel_names.tolist(),
    sfreq=float(fs),
    ch_types="eeg",
)
experiment_info.set_montage(
    mne.channels.make_standard_montage("biosemi128"),
    match_case=False,
    on_missing="raise",
)

assert X.ndim == 3 and X.shape[1] == 128
assert len(X) == len(y) == len(subjects) == len(sessions)
assert np.isfinite(X).all()
assert set(CHANNEL_COUNTS) == set(channel_configs_idx)
assert all(len(np.unique(channel_configs_idx[n])) == n for n in CHANNEL_COUNTS)
assert set(channel_configs_idx[16]).issubset(channel_configs_idx[32])
assert set(channel_configs_idx[32]).issubset(channel_configs_idx[64])
assert set(channel_configs_idx[64]).issubset(channel_configs_idx[128])
validate_subject_split(DEV_SPLIT)

set_all_seeds(42)
print("Run mode:", RUN_MODE)
print("Device:", device)
print("Deep-learning seeds:", ACTIVE_DEEP_SEEDS)
print("Development split:", DEV_SPLIT)
print("Channel hashes:", {n: channel_config_hash(n) for n in CHANNEL_COUNTS})


## Pipeline 1 — rich features + fold-safe MRMR + tuned SVM

The feature bank intentionally follows the paper's categories without claiming an exact reproduction of its unavailable supplementary feature list. It includes time-domain statistics and Hjorth parameters, absolute and relative Welch bandpower, spectral descriptors, and multi-resolution wavelet statistics.

`ApproxMRMR` first keeps the most label-relevant candidates using mutual information and then greedily penalizes correlated selections. Crucially, variance filtering, MRMR, scaling, and SVM tuning live inside the cross-validation pipeline, so validation or test trials cannot influence feature selection.


In [ ]:
BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 45),
}

def evaluate_svm_candidate(
    X_train,
    y_train,
    X_validation,
    candidate,
):
    svm_parameters = {
        key.replace("svm__", ""): value
        for key, value in candidate.items()
    }

    if not GPU_SVM_AVAILABLE:
        model = SVC(**svm_parameters)
        model.fit(X_train, y_train)
        return model.predict(X_validation)

    # Transfer each fold to the GPU once.
    X_train_gpu = cp.asarray(
        X_train,
        dtype=cp.float32,
    )
    X_validation_gpu = cp.asarray(
        X_validation,
        dtype=cp.float32,
    )
    y_train_gpu = cp.asarray(
        y_train,
        dtype=cp.int32,
    )

    classes = np.unique(y_train)
    n_classes = len(classes)
    n_validation = len(X_validation)

    if n_classes < 2:
        raise ValueError(
            "The SVM training fold contains fewer than two classes."
        )

    # Resolve gamma once using the complete inner-training fold.
    # This keeps gamma consistent across all pairwise classifiers.
    gamma = svm_parameters.get("gamma", "scale")

    if gamma == "scale":
        variance = float(cp.var(X_train_gpu).get())

        if variance > 0:
            gamma = 1.0 / (
                X_train.shape[1] * variance
            )
        else:
            gamma = 1.0 / X_train.shape[1]

    elif gamma == "auto":
        gamma = 1.0 / X_train.shape[1]

    kernel = svm_parameters.get("kernel", "rbf")
    C = svm_parameters.get("C", 1.0)

    votes = cp.zeros(
        (n_validation, n_classes),
        dtype=cp.float32,
    )
    confidence_sums = cp.zeros_like(votes)

    # Explicit one-vs-one:
    # four classes produce six binary GPU SVMs.
    for first_index in range(n_classes):
        for second_index in range(
            first_index + 1,
            n_classes,
        ):
            first_class = classes[first_index]
            second_class = classes[second_index]

            pair_mask = (
                (y_train_gpu == first_class)
                | (y_train_gpu == second_class)
            )

            pair_X = X_train_gpu[pair_mask]

            # Encode first class as 0 and second class as 1.
            pair_y = (
                y_train_gpu[pair_mask]
                == second_class
            ).astype(cp.float32)

            binary_model = CuMLSVC(
                kernel=kernel,
                C=C,
                gamma=gamma,
                cache_size=2048,
                output_type="cupy",
            )

            binary_model.fit(
                pair_X,
                pair_y,
            )

            decision = binary_model.decision_function(
                X_validation_gpu
            )

            # Handle either a CuPy array or a cuML array wrapper.
            if hasattr(decision, "to_output"):
                decision = decision.to_output("cupy")

            decision = cp.asarray(
                decision,
                dtype=cp.float32,
            ).reshape(-1)

            predicts_second = decision > 0

            # Accumulate one-vs-one votes.
            votes[:, first_index] += (
                ~predicts_second
            ).astype(cp.float32)

            votes[:, second_index] += (
                predicts_second
            ).astype(cp.float32)

            # Confidence-based tie breaking, following the
            # standard scikit-learn one-vs-one combination.
            confidence_sums[:, first_index] -= decision
            confidence_sums[:, second_index] += decision

            del binary_model, pair_X, pair_y, decision

    transformed_confidence = confidence_sums / (
        3.0 * (cp.abs(confidence_sums) + 1.0)
    )

    combined_scores = (
        votes + transformed_confidence
    )

    predicted_indices = cp.argmax(
        combined_scores,
        axis=1,
    )

    predicted_indices = cp.asnumpy(
        predicted_indices
    )

    return classes[predicted_indices].astype(
        y_train.dtype,
        copy=False,
    )

def extract_rich_feature_tensor(data, sfreq):
    # Return [trial, channel, feature] and per-channel feature names.
    eps = 1e-12
    trial_features = []
    feature_names = None

    for trial in tqdm(data, desc="Rich EEG features", leave=False):
        values = []
        names = []

        first_diff = np.diff(trial, axis=-1)
        second_diff = np.diff(first_diff, axis=-1)
        activity = np.var(trial, axis=-1)
        diff_activity = np.var(first_diff, axis=-1)
        second_activity = np.var(second_diff, axis=-1)
        mobility = np.sqrt(diff_activity / (activity + eps))
        complexity = (
            np.sqrt(second_activity / (diff_activity + eps)) /
            (mobility + eps)
        )

        time_values = [
            np.mean(trial, axis=-1),
            np.std(trial, axis=-1),
            np.sqrt(np.mean(trial ** 2, axis=-1)),
            np.ptp(trial, axis=-1),
            skew(trial, axis=-1, bias=False),
            kurtosis(trial, axis=-1, fisher=True, bias=False),
            np.mean(np.abs(first_diff), axis=-1),
            np.mean(np.diff(np.signbit(trial), axis=-1) != 0, axis=-1),
            activity,
            mobility,
            complexity,
        ]
        time_names = [
            "mean", "std", "rms", "peak_to_peak", "skewness",
            "kurtosis", "line_length", "zero_crossing_rate",
            "hjorth_activity", "hjorth_mobility", "hjorth_complexity",
        ]
        values.extend(time_values)
        names.extend(time_names)

        frequencies, psd = welch(
            trial,
            fs=sfreq,
            nperseg=min(256, trial.shape[-1]),
            axis=-1,
        )
        analysis_mask = (frequencies >= 1) & (frequencies <= 45)
        analysis_psd = psd[:, analysis_mask]
        analysis_freqs = frequencies[analysis_mask]
        total_power = np.trapz(analysis_psd, analysis_freqs, axis=-1) + eps

        for band_name, (low, high) in BANDS.items():
            band_mask = (frequencies >= low) & (frequencies < high)
            band_power = np.trapz(psd[:, band_mask], frequencies[band_mask], axis=-1)
            values.extend([np.log(band_power + eps), band_power / total_power])
            names.extend([f"log_power_{band_name}", f"relative_power_{band_name}"])

        probability = analysis_psd / (analysis_psd.sum(axis=-1, keepdims=True) + eps)
        spectral_entropy = -np.sum(probability * np.log(probability + eps), axis=-1)
        spectral_entropy /= np.log(max(2, analysis_psd.shape[-1]))
        spectral_centroid = (
            (analysis_psd * analysis_freqs[None, :]).sum(axis=-1) /
            (analysis_psd.sum(axis=-1) + eps)
        )
        cumulative_power = np.cumsum(analysis_psd, axis=-1)
        edge_index = (
            cumulative_power >= 0.95 * cumulative_power[:, -1, None]
        ).argmax(axis=-1)
        spectral_edge_95 = analysis_freqs[edge_index]
        values.extend([spectral_entropy, spectral_centroid, spectral_edge_95])
        names.extend(["spectral_entropy", "spectral_centroid", "spectral_edge_95"])

        wavelet_coefficients = pywt.wavedec(trial, "db4", level=4, axis=-1)
        for coefficient_index, coefficient in enumerate(wavelet_coefficients):
            energy = np.mean(coefficient ** 2, axis=-1)
            absolute_mean = np.mean(np.abs(coefficient), axis=-1)
            coefficient_std = np.std(coefficient, axis=-1)
            values.extend([np.log(energy + eps), absolute_mean, coefficient_std])
            names.extend([
                f"wavelet_{coefficient_index}_log_energy",
                f"wavelet_{coefficient_index}_abs_mean",
                f"wavelet_{coefficient_index}_std",
            ])

        matrix = np.stack(values, axis=-1)
        trial_features.append(np.nan_to_num(matrix, copy=False))
        if feature_names is None:
            feature_names = names

    return np.asarray(trial_features, dtype=np.float32), feature_names


RICH_FEATURE_CACHE = CACHE_DIR / (
    f"rich_features_{len(X)}x{X.shape[1]}x{X.shape[2]}_{float(fs):g}hz.npz"
)


def get_rich_feature_tensor():
    if RICH_FEATURE_CACHE.exists():
        cached = np.load(RICH_FEATURE_CACHE, allow_pickle=False)
        tensor = cached["features"]
        names = cached["feature_names"].tolist()
        expected = (len(X), X.shape[1])
        if tensor.shape[:2] != expected:
            raise ValueError(f"Stale feature cache: {tensor.shape[:2]} != {expected}")
        return tensor, names

    tensor, names = extract_rich_feature_tensor(X, fs)
    np.savez_compressed(
        RICH_FEATURE_CACHE,
        features=tensor,
        feature_names=np.asarray(names),
    )
    return tensor, names


class ApproxMRMR(BaseEstimator, TransformerMixin):
    # Efficient relevance-minus-redundancy selector fitted on training data only.

    def __init__(self, n_features=590, prefilter=2000, redundancy_weight=1.0, random_state=42):
        self.n_features = n_features
        self.prefilter = prefilter
        self.redundancy_weight = redundancy_weight
        self.random_state = random_state

    def fit(self, X_fit, y_fit):
        X_fit = np.asarray(X_fit, dtype=np.float32)
        relevance = mutual_info_classif(
            X_fit,
            y_fit,
            random_state=self.random_state,
        )
        candidate_count = min(int(self.prefilter), X_fit.shape[1])
        candidates = np.argsort(relevance)[::-1][:candidate_count]
        candidate_relevance = relevance[candidates]
        relevance_scale = candidate_relevance.max()
        if relevance_scale > 0:
            candidate_relevance = candidate_relevance / relevance_scale

        candidate_data = X_fit[:, candidates]
        correlation = np.abs(np.corrcoef(candidate_data, rowvar=False))
        correlation = np.nan_to_num(correlation, copy=False).astype(np.float32)
        np.fill_diagonal(correlation, 0.0)

        selection_count = min(int(self.n_features), candidate_count)
        selected = []
        available = np.ones(candidate_count, dtype=bool)
        redundancy_sum = np.zeros(candidate_count, dtype=np.float32)

        for step in range(selection_count):
            mean_redundancy = redundancy_sum / max(1, step)
            score = candidate_relevance - self.redundancy_weight * mean_redundancy
            score[~available] = -np.inf
            chosen = int(np.argmax(score))
            selected.append(chosen)
            available[chosen] = False
            redundancy_sum += correlation[:, chosen]

        self.selected_indices_ = candidates[np.asarray(selected, dtype=int)]
        self.relevance_ = relevance
        self.n_features_in_ = X_fit.shape[1]
        return self

    def transform(self, X_transform):
        return np.asarray(X_transform)[:, self.selected_indices_]


In [ ]:
def rich_features_for_channels(n_channels):
    tensor, base_names = get_rich_feature_tensor()
    channel_indices = np.asarray(
        channel_configs_idx[n_channels],
        dtype=int,
    )

    features = tensor[:, channel_indices, :].reshape(len(X), -1)

    names = np.asarray([
        f"{channel_names[channel_index]}::{feature_name}"
        for channel_index in channel_indices
        for feature_name in base_names
    ])

    return features, names


def make_svm_estimator(seed=42, final_search=False):
    selected_count = 590
    prefilter = 3000 if final_search else 1500

    # Do not use Pipeline(memory=...).
    # Caching notebook-defined transformers causes the PicklingError.
    estimator = Pipeline([
        ("variance", VarianceThreshold()),
        (
            "mrmr",
            ApproxMRMR(
                n_features=selected_count,
                prefilter=prefilter,
                random_state=seed,
            ),
        ),
        ("scaler", StandardScaler()),
        ("svm", SVC()),
    ])

    if final_search:
        parameter_grid = [
            {
                "svm__kernel": ["linear"],
                "svm__C": [0.1, 1, 10, 100],
            },
            {
                "svm__kernel": ["rbf"],
                "svm__C": [0.1, 1, 10, 100],
                "svm__gamma": [
                    "scale",
                    1e-3,
                    1e-2,
                    1e-1,
                ],
            },
        ]
    else:
        parameter_grid = [
            {
                "svm__kernel": ["linear"],
                "svm__C": [1, 10],
            },
            {
                "svm__kernel": ["rbf"],
                "svm__C": [1, 10],
                "svm__gamma": ["scale", 1e-2],
            },
        ]

    return estimator, parameter_grid


def tune_fold_safe_svm(
    features,
    labels,
    seed=42,
    groups=None,
    grouped=True,
    inner_folds=None,
):
    """
    Tune SVM hyperparameters while fitting variance filtering,
    MRMR and scaling once per inner training fold.
    """
    base_estimator, parameter_grid = make_svm_estimator(
        seed=seed,
        final_search=(RUN_MODE == "final"),
    )

    candidates = list(ParameterGrid(parameter_grid))
    inner_folds = inner_folds or (
        5 if RUN_MODE == "final" else 3
    )

    if grouped:
        if groups is None:
            raise ValueError(
                "groups are required for grouped inner CV"
            )

        splitter = StratifiedGroupKFold(
            n_splits=inner_folds,
            shuffle=True,
            random_state=seed,
        )

        splits = splitter.split(
            features,
            labels,
            groups,
        )
    else:
        splitter = StratifiedKFold(
            n_splits=inner_folds,
            shuffle=True,
            random_state=seed,
        )

        splits = splitter.split(
            features,
            labels,
        )

    fold_scores = [[] for _ in candidates]

    for inner_fold, (
        train_index,
        validation_index,
    ) in enumerate(splits, start=1):

        # Fit all data-dependent preprocessing using only
        # this inner fold's training partition.
        variance = clone(
            base_estimator.named_steps["variance"]
        )
        selector = clone(
            base_estimator.named_steps["mrmr"]
        )
        scaler = clone(
            base_estimator.named_steps["scaler"]
        )

        fold_train = variance.fit_transform(
            features[train_index],
            labels[train_index],
        )
        fold_validation = variance.transform(
            features[validation_index]
        )

        fold_train = selector.fit_transform(
            fold_train,
            labels[train_index],
        )
        fold_validation = selector.transform(
            fold_validation
        )

        fold_train = scaler.fit_transform(fold_train)
        fold_validation = scaler.transform(
            fold_validation
        )

        # Reuse the transformed arrays for every SVM candidate.
        for candidate_index, candidate in enumerate(
            candidates
        ):
            prediction = evaluate_svm_candidate(
                fold_train,
                labels[train_index],
                fold_validation,
                candidate,
            )

            score = balanced_accuracy_score(
                labels[validation_index],
                prediction,
            )

            fold_scores[candidate_index].append(score)

    mean_scores = np.asarray([
        np.mean(scores)
        for scores in fold_scores
    ])

    best_index = int(np.argmax(mean_scores))
    best_params = candidates[best_index]
    best_score = float(mean_scores[best_index])

    # Refit the complete pipeline on all development-training data.
    final_estimator = clone(
        base_estimator
    ).set_params(**best_params)

    final_estimator.fit(
        features,
        labels,
    )

    cv_results = pd.DataFrame([
        {
            "params": json.dumps(
                candidate,
                sort_keys=True,
            ),
            "mean_balanced_accuracy": mean_scores[index],
            "std_balanced_accuracy": np.std(
                fold_scores[index]
            ),
            "fold_scores": json.dumps(
                fold_scores[index]
            ),
        }
        for index, candidate in enumerate(candidates)
    ]).sort_values(
        "mean_balanced_accuracy",
        ascending=False,
    )

    return (
        final_estimator,
        best_params,
        best_score,
        cv_results,
    )


def run_rich_svm_experiment(
    n_channels,
    split=DEV_SPLIT,
    protocol="development",
    fold="fixed",
    seed=42,
):
    masks = subject_split_masks(split)

    features, feature_names = (
        rich_features_for_channels(n_channels)
    )

    train_groups = subjects[masks["train"]]
    inner_folds = 5 if RUN_MODE == "final" else 3

    (
        estimator,
        best_params,
        best_inner_score,
        cv_results,
    ) = tune_fold_safe_svm(
        features[masks["train"]],
        y[masks["train"]],
        groups=train_groups,
        grouped=True,
        inner_folds=inner_folds,
        seed=seed,
    )

    cv_results.to_csv(
        RESULTS_DIR
        / (
            f"{protocol}_{fold}_svm_"
            f"{n_channels}ch_inner_cv.csv"
        ),
        index=False,
    )

    val_pred = estimator.predict(
        features[masks["val"]]
    )
    test_pred = estimator.predict(
        features[masks["test"]]
    )

    val_metrics = classification_metrics(
        y[masks["val"]],
        val_pred,
    )
    test_metrics = classification_metrics(
        y[masks["test"]],
        test_pred,
    )

    variance_indices = (
        estimator
        .named_steps["variance"]
        .get_support(indices=True)
    )
    mrmr_indices = (
        estimator
        .named_steps["mrmr"]
        .selected_indices_
    )

    selected_names = (
        feature_names[variance_indices][mrmr_indices]
    )

    selected_path = RESULTS_DIR / (
        f"{protocol}_{fold}_svm_"
        f"{n_channels}ch_selected_features.txt"
    )
    selected_path.write_text(
        "\n".join(selected_names.tolist()) + "\n"
    )

    store_predictions(
        protocol,
        fold,
        "SVM_rich_MRMR",
        n_channels,
        "val",
        masks["val"],
        val_pred,
        seed,
    )
    store_predictions(
        protocol,
        fold,
        "SVM_rich_MRMR",
        n_channels,
        "test",
        masks["test"],
        test_pred,
        seed,
    )

    row = store_result(
        protocol,
        fold,
        "SVM_rich_MRMR",
        n_channels,
        split,
        val_metrics,
        test_metrics,
        seed=seed,
        config_extra={
            "representation":
                "time + Welch + wavelet features",
            "normalization":
                "StandardScaler inside inner CV",
            "feature_selection":
                "ApproxMRMR inside inner CV",
            "selected_feature_count":
                len(selected_names),
            "inner_cv":
                (
                    f"{inner_folds}-fold "
                    "StratifiedGroupKFold by subject"
                ),
            "best_params":
                json.dumps(
                    best_params,
                    sort_keys=True,
                ),
            "best_inner_balanced_accuracy":
                best_inner_score,
            "checkpoint": None,
        },
    )

    print(
        f"Rich SVM | {n_channels:3d} ch | "
        f"val bal="
        f"{val_metrics['balanced_accuracy']:.3f} | "
        f"test bal="
        f"{test_metrics['balanced_accuracy']:.3f} | "
        f"{best_params}"
    )

    return estimator, row

In [ ]:
rich_svm_models = {}
if RUN_DEVELOPMENT_RICH_SVM:
    for n_channels in CHANNEL_COUNTS:
        rich_svm_models[n_channels], _ = run_rich_svm_experiment(n_channels)
else:
    print("Rich SVM development runs are disabled.")


## Shared deep-learning utilities

EEGNet uses training-only channel normalization. Its augmentation is deliberately mild and is applied only while sampling the training set. LaBraM instead uses the checkpoint-compatible physical scaling described in its downstream literature.

Early stopping monitors **validation balanced accuracy**, restores the best checkpoint, and therefore may correctly stop before the requested maximum epoch.


In [ ]:
class EEGArrayDataset(Dataset):
    def __init__(self, data, labels, augment=False, seed=42):
        self.X = torch.from_numpy(np.ascontiguousarray(data, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(labels, dtype=np.int64))
        self.augment = augment
        self.generator = torch.Generator().manual_seed(seed)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        signal = self.X[index].clone()
        if self.augment:
            amplitude = 0.9 + 0.2 * torch.rand((), generator=self.generator)
            signal *= amplitude
            if torch.rand((), generator=self.generator) < 0.5:
                shift = int(torch.randint(-16, 17, (), generator=self.generator))
                signal = torch.roll(signal, shifts=shift, dims=-1)
            noise_scale = 0.01 * signal.std().clamp_min(1e-6)
            noise = torch.randn(signal.shape, generator=self.generator) * noise_scale
            signal += noise
            if torch.rand((), generator=self.generator) < 0.25:
                drop_count = max(1, int(round(0.05 * signal.shape[0])))
                dropped = torch.randperm(signal.shape[0], generator=self.generator)[:drop_count]
                signal[dropped] = 0.0
        return signal, self.y[index]


def resample_array(data, source_sfreq, target_sfreq):
    if np.isclose(source_sfreq, target_sfreq):
        return np.asarray(data, dtype=np.float32)
    ratio = Fraction(target_sfreq / source_sfreq).limit_denominator(1000)
    return resample_poly(
        data,
        up=ratio.numerator,
        down=ratio.denominator,
        axis=-1,
    ).astype(np.float32, copy=False)


def prepare_eegnet_splits(n_channels, split):
    masks = subject_split_masks(split)
    indices = np.asarray(channel_configs_idx[n_channels], dtype=int)
    data = np.asarray(X[:, indices, :], dtype=np.float32)
    training_data = data[masks["train"]]
    mean = training_data.mean(axis=(0, 2), keepdims=True, dtype=np.float64)
    std = training_data.std(axis=(0, 2), keepdims=True, dtype=np.float64)
    std = np.maximum(std, 1e-8)
    arrays = {
        name: ((data[mask] - mean) / std).astype(np.float32)
        for name, mask in masks.items()
    }
    labels = {name: y[mask] for name, mask in masks.items()}
    return arrays, labels, masks


def make_loaders(arrays, labels, batch_size, seed, augment_train=False):
    generator = torch.Generator().manual_seed(seed)
    return {
        "train": DataLoader(
            EEGArrayDataset(arrays["train"], labels["train"], augment=augment_train, seed=seed),
            batch_size=batch_size,
            shuffle=True,
            generator=generator,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        ),
        "val": DataLoader(
            EEGArrayDataset(arrays["val"], labels["val"]),
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        ),
        "test": DataLoader(
            EEGArrayDataset(arrays["test"], labels["test"]),
            batch_size=batch_size,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available(),
        ),
    }


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    labels = []
    predictions = []
    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * len(batch_y)
        labels.append(batch_y.detach().cpu().numpy())
        predictions.append(logits.argmax(1).detach().cpu().numpy())
    labels = np.concatenate(labels)
    predictions = np.concatenate(predictions)
    return total_loss / len(labels), classification_metrics(labels, predictions)


@torch.no_grad()
def predict_torch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    labels = []
    predictions = []
    for batch_X, batch_y in loader:
        batch_X = batch_X.to(device, non_blocking=True)
        batch_y = batch_y.to(device, non_blocking=True)
        logits = model(batch_X)
        total_loss += criterion(logits, batch_y).item() * len(batch_y)
        labels.append(batch_y.cpu().numpy())
        predictions.append(logits.argmax(1).cpu().numpy())
    labels = np.concatenate(labels)
    predictions = np.concatenate(predictions)
    return total_loss / len(labels), labels, predictions


def fit_torch_stage(
    model,
    loaders,
    optimizer,
    epochs,
    patience,
    checkpoint_path,
    description,
    scheduler=None,
):
    criterion = nn.CrossEntropyLoss()
    best_score = -np.inf
    best_loss = np.inf
    best_epoch = 0
    best_state = None
    history = []
    epochs_without_improvement = 0

    progress = tqdm(range(1, epochs + 1), desc=description)
    for epoch in progress:
        train_loss, train_metrics = train_one_epoch(
            model, loaders["train"], optimizer, criterion
        )
        val_loss, val_y, val_pred = predict_torch(model, loaders["val"], criterion)
        val_metrics = classification_metrics(val_y, val_pred)
        current_lr = max(group["lr"] for group in optimizer.param_groups)
        history.append({
            "epoch": epoch,
            "learning_rate": current_lr,
            "train_loss": train_loss,
            **{f"train_{key}": value for key, value in train_metrics.items()},
            "val_loss": val_loss,
            **{f"val_{key}": value for key, value in val_metrics.items()},
        })

        improved = (
            val_metrics["balanced_accuracy"] > best_score + 1e-12 or
            (
                np.isclose(val_metrics["balanced_accuracy"], best_score) and
                val_loss < best_loss - 1e-12
            )
        )
        if improved:
            best_score = val_metrics["balanced_accuracy"]
            best_loss = val_loss
            best_epoch = epoch
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if scheduler is not None:
            scheduler.step()

        progress.set_postfix(
            train=f"{train_metrics['balanced_accuracy']:.3f}",
            val=f"{val_metrics['balanced_accuracy']:.3f}",
            best=f"{best_score:.3f}",
        )
        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        raise RuntimeError("Training ended without a valid checkpoint.")
    model.load_state_dict(best_state)
    torch.save(best_state, checkpoint_path)
    return pd.DataFrame(history), best_epoch


def finalize_torch_experiment(
    model,
    loaders,
    masks,
    split,
    protocol,
    fold,
    model_name,
    n_channels,
    seed,
    best_epoch,
    checkpoint_path,
    config_extra,
):
    criterion = nn.CrossEntropyLoss()
    _, val_y, val_pred = predict_torch(model, loaders["val"], criterion)
    _, test_y, test_pred = predict_torch(model, loaders["test"], criterion)
    val_metrics = classification_metrics(val_y, val_pred)
    test_metrics = classification_metrics(test_y, test_pred)

    store_predictions(
        protocol, fold, model_name, n_channels,
        "val", masks["val"], val_pred, seed,
    )
    store_predictions(
        protocol, fold, model_name, n_channels,
        "test", masks["test"], test_pred, seed,
    )
    row = store_result(
        protocol,
        fold,
        model_name,
        n_channels,
        split,
        val_metrics,
        test_metrics,
        seed=seed,
        best_epoch=best_epoch,
        config_extra={**config_extra, "checkpoint": str(checkpoint_path)},
    )
    print(
        f"{model_name} | seed={seed} | {n_channels:3d} ch | "
        f"val bal={val_metrics['balanced_accuracy']:.3f} | "
        f"test bal={test_metrics['balanced_accuracy']:.3f}"
    )
    return row


## Pipeline 2 — improved EEGNet from scratch

The maximum epoch count is a ceiling, not a requirement. Training stops when validation balanced accuracy fails to improve for the configured patience and then restores the best epoch. In `final` mode, every channel condition is repeated with five deterministic seeds.


In [ ]:
def run_eegnet_experiment(
    n_channels,
    split=DEV_SPLIT,
    protocol="development",
    fold="fixed",
    seed=42,
    epochs=None,
    patience=None,
    batch_size=32,
):
    epochs = epochs or (60 if RUN_MODE == "quick" else 120)
    patience = patience or (12 if RUN_MODE == "quick" else 20)
    set_all_seeds(seed)
    arrays, labels, masks = prepare_eegnet_splits(n_channels, split)
    loaders = make_loaders(
        arrays,
        labels,
        batch_size=batch_size,
        seed=seed,
        augment_train=True,
    )
    model = EEGNet(n_channels=n_channels, n_classes=4).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-5
    )
    checkpoint_path = CHECKPOINT_DIR / (
        f"{protocol}_{fold}_eegnet_{n_channels}ch_seed{seed}.pt"
    )
    history, best_epoch = fit_torch_stage(
        model,
        loaders,
        optimizer,
        epochs=epochs,
        patience=patience,
        checkpoint_path=checkpoint_path,
        description=f"EEGNet {n_channels}ch seed{seed}",
        scheduler=scheduler,
    )
    history.assign(
        protocol=protocol,
        fold=fold,
        model="EEGNet_augmented",
        n_channels=n_channels,
        seed=seed,
    ).to_csv(
        RESULTS_DIR / f"{protocol}_{fold}_eegnet_{n_channels}ch_seed{seed}_history.csv",
        index=False,
    )
    row = finalize_torch_experiment(
        model,
        loaders,
        masks,
        split,
        protocol,
        fold,
        "EEGNet_augmented",
        n_channels,
        seed,
        best_epoch,
        checkpoint_path,
        config_extra={
            "representation": "raw time-domain EEG",
            "normalization": "per-channel z-score fit on train only",
            "augmentation": "amplitude + shift + Gaussian noise + channel dropout; train only",
            "model_sfreq": float(fs),
            "epochs_requested": epochs,
            "early_stopping_patience": patience,
            "batch_size": batch_size,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "learning_rate": 1e-3,
        },
    )
    return model, row


In [ ]:
eegnet_models = {}
if RUN_DEVELOPMENT_EEGNET:
    for seed in ACTIVE_DEEP_SEEDS:
        for n_channels in CHANNEL_COUNTS:
            eegnet_models[(seed, n_channels)], _ = run_eegnet_experiment(
                n_channels,
                seed=seed,
            )
else:
    print("EEGNet development runs are disabled.")


## Pipeline 3 — pretrained LaBraM-Base

Install the checkpoint integration once in Kaggle, restart the kernel, and enable the run flag:

```python
%pip install -q "braindecode[hub]>=1.8,<2"
```

BioSemi-128 uses `A1…D32` sensor labels, whereas LaBraM learned a different canonical 128-position montage. `InterpolatedLaBraM` therefore builds a fixed MNE spherical-spline projection with shape **128 × input_channels**. The projection is determined only by sensor geometry and remains frozen. This is still materially less restrictive than BIOT's projection to only 18 targets, but it must be reported as an experimental montage adaptation.

The 2.5-second action window becomes 500 samples at 200 Hz. It is symmetrically zero-padded after physical scaling to 600 samples so LaBraM receives three complete 200-sample patches. Pretrained weights are adapted from the released 3000-sample checkpoint by interpolating only its temporal-position embedding; all compatible encoder tensors retain their pretrained values.

References: [LaBraM paper and official code](https://github.com/935963004/LaBraM), [Braindecode checkpoint](https://huggingface.co/braindecode/labram-pretrained), and [arbitrary-montage adaptation guide](https://braindecode.org/dev/auto_examples/model_building/plot_channel_interpolation.html).


In [ ]:
LABRAM_REPO_ID = "braindecode/labram-pretrained"
LABRAM_SFREQ = 200
LABRAM_PATCH_SIZE = 200


def prepare_labram_splits(n_channels, split):
    masks = subject_split_masks(split)
    indices = np.asarray(channel_configs_idx[n_channels], dtype=int)
    data = np.asarray(X[:, indices, :], dtype=np.float32)
    data = resample_array(data, float(fs), LABRAM_SFREQ)

    # MNE epochs are in volts. Convert to microvolts, then divide by
    # 100 to match the 0.1 mV scaling used in LaBraM downstream work.
    data = data * 1e6 / 100.0

    remainder = data.shape[-1] % LABRAM_PATCH_SIZE
    if remainder:
        padding = LABRAM_PATCH_SIZE - remainder
        left = padding // 2
        right = padding - left
        data = np.pad(
            data,
            ((0, 0), (0, 0), (left, right)),
            mode="constant",
            constant_values=0.0,
        )

    arrays = {
        name: np.asarray(data[mask], dtype=np.float32)
        for name, mask in masks.items()
    }
    labels = {name: y[mask] for name, mask in masks.items()}
    return arrays, labels, masks


def _interpolate_temporal_embedding(source_tensor, target_shape):
    if tuple(source_tensor.shape) == tuple(target_shape):
        return source_tensor
    resized = torch.nn.functional.interpolate(
        source_tensor.transpose(1, 2),
        size=target_shape[1],
        mode="linear",
        align_corners=False,
    ).transpose(1, 2)
    return resized


def load_adapted_labram(n_channels, n_times, pretrained=True):
    try:
        import braindecode
        from braindecode.models import InterpolatedLaBraM
    except (ImportError, AttributeError) as exc:
        raise RuntimeError(
            'LaBraM montage adaptation requires Braindecode. Run '
            '`%pip install -q "braindecode[hub]>=1.8,<2"`, restart '
            "the kernel, and rerun the notebook."
        ) from exc

    indices = np.asarray(channel_configs_idx[n_channels], dtype=int)
    subset_info = mne.pick_info(experiment_info.copy(), indices.tolist())
    chs_info = subset_info["chs"]

    target_model = InterpolatedLaBraM(
        chs_info=chs_info,
        n_times=n_times,
        n_outputs=4,
        patch_size=LABRAM_PATCH_SIZE,
        sfreq=LABRAM_SFREQ,
        trainable=False,
    )
    load_report = {
        "pretrained": pretrained,
        "copied_tensors": 0,
        "adapted_tensors": [],
        "skipped_tensors": [],
    }

    if pretrained:
        try:
            source_model = InterpolatedLaBraM.from_pretrained(
                LABRAM_REPO_ID,
                chs_info=chs_info,
                n_times=3000,
                n_outputs=4,
                patch_size=LABRAM_PATCH_SIZE,
                strict=False,
                trainable=False,
            )
        except Exception as exc:
            raise RuntimeError(
                f"Could not load {LABRAM_REPO_ID}. Enable Kaggle Internet "
                "once or attach a populated Hugging Face cache. Random "
                "initialization is never substituted silently."
            ) from exc

        source_state = source_model.state_dict()
        target_state = target_model.state_dict()
        adapted_state = {}
        for name, target_value in target_state.items():
            if name.startswith("final_layer"):
                load_report["skipped_tensors"].append(name)
                continue
            if name not in source_state:
                load_report["skipped_tensors"].append(name)
                continue
            source_value = source_state[name]
            if tuple(source_value.shape) == tuple(target_value.shape):
                adapted_state[name] = source_value
                load_report["copied_tensors"] += 1
            elif name.endswith("temporal_embedding"):
                adapted_state[name] = _interpolate_temporal_embedding(
                    source_value,
                    target_value.shape,
                )
                load_report["adapted_tensors"].append(name)
            else:
                load_report["skipped_tensors"].append(name)

        target_model.load_state_dict(adapted_state, strict=False)
        del source_model

    projection_shape = tuple(target_model.interpolation_layer.matrix.shape)
    expected_projection = (128, n_channels)
    if projection_shape != expected_projection:
        raise RuntimeError(
            f"Unexpected LaBraM projection {projection_shape}; "
            f"expected {expected_projection}."
        )
    return target_model, braindecode.__version__, projection_shape, load_report


def configure_labram_stage(model, stage):
    for parameter in model.parameters():
        parameter.requires_grad = False

    if stage == "head":
        modules = [model.final_layer]
    elif stage == "partial":
        modules = [*model.blocks[-4:], model.norm, model.final_layer]
        if model.fc_norm is not None:
            modules.append(model.fc_norm)
    elif stage == "full":
        for parameter in model.parameters():
            parameter.requires_grad = True
        modules = []
    else:
        raise ValueError(f"Unknown LaBraM stage: {stage}")

    for module in modules:
        for parameter in module.parameters():
            parameter.requires_grad = True

    # The geometry-derived projection must remain fixed in all stages.
    for parameter in model.interpolation_layer.parameters():
        parameter.requires_grad = False


def make_labram_optimizer(model, stage):
    head_parameters = [
        parameter for parameter in model.final_layer.parameters()
        if parameter.requires_grad
    ]
    encoder_parameters = [
        parameter
        for name, parameter in model.named_parameters()
        if parameter.requires_grad and not name.startswith("final_layer")
    ]
    parameter_groups = []
    if encoder_parameters:
        parameter_groups.append({"params": encoder_parameters, "lr": 1e-5})
    if head_parameters:
        parameter_groups.append({"params": head_parameters, "lr": 1e-3 if stage == "head" else 3e-4})
    return torch.optim.AdamW(parameter_groups, weight_decay=0.05)


In [ ]:
def run_labram_experiment(
    n_channels,
    split=DEV_SPLIT,
    protocol="development",
    fold="fixed",
    seed=42,
    pretrained=True,
    batch_size=16,
):
    set_all_seeds(seed)
    arrays, labels, masks = prepare_labram_splits(n_channels, split)
    loaders = make_loaders(
        arrays,
        labels,
        batch_size=batch_size,
        seed=seed,
        augment_train=False,
    )
    model, braindecode_version, projection_shape, load_report = load_adapted_labram(
        n_channels,
        n_times=arrays["train"].shape[-1],
        pretrained=pretrained,
    )
    model = model.to(device)
    model_name = "LaBraM_pretrained" if pretrained else "LaBraM_random_init"

    if pretrained:
        stages = [
            ("head", 10, 6),
            ("partial", 25, 10),
            ("full", 40, 12),
        ]
    else:
        stages = [("full", 80, 15)]

    histories = []
    best_epochs = {}
    final_checkpoint = None
    for stage, epochs, patience in stages:
        configure_labram_stage(model, stage)
        optimizer = make_labram_optimizer(model, stage)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
            eta_min=1e-6,
        )
        checkpoint_path = CHECKPOINT_DIR / (
            f"{protocol}_{fold}_{model_name}_{n_channels}ch_"
            f"seed{seed}_{stage}.pt"
        )
        history, best_epoch = fit_torch_stage(
            model,
            loaders,
            optimizer,
            epochs=epochs,
            patience=patience,
            checkpoint_path=checkpoint_path,
            description=f"{model_name} {stage} {n_channels}ch seed{seed}",
            scheduler=scheduler,
        )
        histories.append(history.assign(stage=stage))
        best_epochs[stage] = best_epoch
        final_checkpoint = checkpoint_path

    pd.concat(histories, ignore_index=True).assign(
        protocol=protocol,
        fold=fold,
        model=model_name,
        n_channels=n_channels,
        seed=seed,
    ).to_csv(
        RESULTS_DIR / (
            f"{protocol}_{fold}_{model_name}_{n_channels}ch_"
            f"seed{seed}_history.csv"
        ),
        index=False,
    )

    row = finalize_torch_experiment(
        model,
        loaders,
        masks,
        split,
        protocol,
        fold,
        model_name,
        n_channels,
        seed,
        best_epochs[stages[-1][0]],
        final_checkpoint,
        config_extra={
            "representation": "raw EEG projected to LaBraM canonical 128-position montage",
            "normalization": "volts -> microvolts / 100; no test-fitted statistics",
            "model_sfreq": LABRAM_SFREQ,
            "model_n_times": arrays["train"].shape[-1],
            "pretrained_repo": LABRAM_REPO_ID if pretrained else None,
            "pretrained": pretrained,
            "braindecode_version": braindecode_version,
            "projection_shape": str(projection_shape),
            "projection_trainable": False,
            "load_report": json.dumps(load_report, sort_keys=True),
            "stage_best_epochs": json.dumps(best_epochs, sort_keys=True),
            "batch_size": batch_size,
        },
    )
    return model, row


In [ ]:
labram_models = {}
if RUN_DEVELOPMENT_LABRAM:
    for seed in ACTIVE_DEEP_SEEDS:
        for n_channels in CHANNEL_COUNTS:
            labram_models[("pretrained", seed, n_channels)], _ = run_labram_experiment(
                n_channels,
                seed=seed,
                pretrained=True,
            )
            if RUN_LABRAM_RANDOM_CONTROL:
                labram_models[("random", seed, n_channels)], _ = run_labram_experiment(
                    n_channels,
                    seed=seed,
                    pretrained=False,
                )
else:
    print(
        "LaBraM is ready but disabled. Install Braindecode/checkpoint access, "
        "set RUN_DEVELOPMENT_LABRAM = True, and rerun this cell."
    )


## Results and diagnostics

The result table preserves every seed instead of silently choosing the best run. The summary reports mean and standard deviation across seeds. Accuracy retention is measured against the same model and seed at 128 channels.


In [ ]:
def build_results_tables():
    results = pd.DataFrame(EXPERIMENT_ROWS.values())
    predictions = (
        pd.concat(PREDICTION_FRAMES.values(), ignore_index=True)
        if PREDICTION_FRAMES
        else pd.DataFrame()
    )
    if results.empty:
        return results, predictions, pd.DataFrame(), pd.DataFrame()

    results = results.sort_values(
        ["protocol", "fold", "model", "seed", "n_channels"],
        ascending=[True, True, True, True, False],
    ).reset_index(drop=True)

    full_channel_accuracy = (
        results[results["n_channels"] == 128]
        .set_index(["protocol", "fold", "model", "seed"])["test_accuracy"]
    )
    results["accuracy_retention_percent"] = [
        100.0 * row.test_accuracy /
        full_channel_accuracy.get(
            (row.protocol, row.fold, row.model, row.seed), np.nan
        )
        for row in results.itertuples()
    ]

    if predictions.empty:
        subject_metrics = pd.DataFrame()
    else:
        subject_metrics = (
            predictions[predictions["partition"] == "test"]
            .groupby(
                ["protocol", "fold", "model", "n_channels", "seed", "subject"],
                as_index=False,
            )
            .apply(
                lambda frame: pd.Series(
                    classification_metrics(frame["y_true"], frame["y_pred"])
                ),
            )
            .reset_index(drop=True)
        )

    metric_columns = [
        "test_accuracy",
        "test_balanced_accuracy",
        "test_macro_f1",
        "accuracy_retention_percent",
    ]
    summary = (
        results
        .groupby(["protocol", "model", "n_channels"])[metric_columns]
        .agg(["mean", "std", "count"])
    )

    results.to_csv(RESULTS_DIR / "channel_reduction_results.csv", index=False)
    predictions.to_csv(RESULTS_DIR / "trial_predictions.csv", index=False)
    subject_metrics.to_csv(RESULTS_DIR / "subject_level_metrics.csv", index=False)
    summary.to_csv(RESULTS_DIR / "summary_mean_std.csv")
    return results, predictions, subject_metrics, summary


results_df, predictions_df, subject_metrics_df, summary_df = build_results_tables()
display(results_df)
display(summary_df)


def plot_development_confusions(predictions=predictions_df):
    if predictions.empty:
        print("No predictions are available yet.")
        return

    data = predictions[
        (predictions["protocol"] == "development") &
        (predictions["fold"] == "fixed") &
        (predictions["partition"] == "test")
    ]
    for (model_name, seed), model_data in data.groupby(["model", "seed"]):
        fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
        for ax, n_channels in zip(axes, CHANNEL_COUNTS):
            subset = model_data[model_data["n_channels"] == n_channels]
            matrix = confusion_matrix(
                subset["y_true"],
                subset["y_pred"],
                labels=range(4),
                normalize="true",
            )
            image = ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
            ax.set_title(f"{n_channels} channels")
            ax.set_xticks(range(4), CLASS_NAMES.values(), rotation=45)
            ax.set_yticks(range(4), CLASS_NAMES.values())
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
        fig.colorbar(image, ax=axes, shrink=0.75)
        fig.suptitle(f"{model_name}, seed {seed}: normalized test confusion")
        plt.show()


plot_development_confusions()


## Optional final evaluations

`run_loso` is the primary unseen-subject evaluation. Each fold uses one subject for testing, a different subject for validation/early stopping, and the remaining eight for training.

`run_paper_comparable_svm` runs subject-specific stratified 10-fold outer evaluation with a nested inner SVM search. It is the closest rigorous comparison with the paper's high subject-specific numbers. MRMR and all preprocessing are refitted inside every outer training fold.

These are not launched by default because the complete matrix is computationally expensive.


In [ ]:
def make_loso_splits():
    unique_subjects = sorted(np.unique(subjects).tolist())
    folds = []
    for position, test_subject in enumerate(unique_subjects):
        val_subject = unique_subjects[(position + 1) % len(unique_subjects)]
        train_subjects = tuple(
            subject
            for subject in unique_subjects
            if subject not in (test_subject, val_subject)
        )
        folds.append({
            "fold": f"test_subject_{test_subject:02d}",
            "split": {
                "train": train_subjects,
                "val": (val_subject,),
                "test": (test_subject,),
            },
        })
    return folds


def run_loso(
    pipelines=("svm",),
    channel_counts=CHANNEL_COUNTS,
    deep_seeds=(42,),
):
    allowed = {"svm", "eegnet", "labram"}
    unknown = set(pipelines) - allowed
    if unknown:
        raise ValueError(f"Unknown pipelines: {sorted(unknown)}")

    for fold_spec in make_loso_splits():
        fold = fold_spec["fold"]
        split = fold_spec["split"]
        print(f"\n===== {fold}: {split} =====")
        for n_channels in channel_counts:
            if "svm" in pipelines:
                run_rich_svm_experiment(
                    n_channels,
                    split,
                    protocol="loso",
                    fold=fold,
                    seed=42,
                )
            for seed in deep_seeds:
                if "eegnet" in pipelines:
                    run_eegnet_experiment(
                        n_channels,
                        split,
                        protocol="loso",
                        fold=fold,
                        seed=seed,
                    )
                if "labram" in pipelines:
                    run_labram_experiment(
                        n_channels,
                        split,
                        protocol="loso",
                        fold=fold,
                        seed=seed,
                        pretrained=True,
                    )
        build_results_tables()


def run_paper_comparable_svm(
    channel_counts=(128,),
    subject_ids=None,
    seed=42,
):
    feature_results = []
    subject_ids = (
        sorted(np.unique(subjects).tolist())
        if subject_ids is None
        else list(subject_ids)
    )

    for n_channels in channel_counts:
        all_features, _ = rich_features_for_channels(n_channels)
        for subject_id in subject_ids:
            subject_mask = subjects == subject_id
            subject_features = all_features[subject_mask]
            subject_labels = y[subject_mask]
            minimum_class_count = int(np.bincount(subject_labels).min())
            outer_splits = min(10, minimum_class_count)
            outer_cv = StratifiedKFold(
                n_splits=outer_splits,
                shuffle=True,
                random_state=seed,
            )

            for outer_fold, (train_index, test_index) in enumerate(
                outer_cv.split(subject_features, subject_labels),
                start=1,
            ):
                estimator, best_params, best_inner_score, _ = tune_fold_safe_svm(
                    subject_features[train_index],
                    subject_labels[train_index],
                    seed=seed + outer_fold,
                    groups=None,
                    grouped=False,
                    inner_folds=3,
                )
                prediction = estimator.predict(subject_features[test_index])
                metrics = classification_metrics(
                    subject_labels[test_index],
                    prediction,
                )
                feature_results.append({
                    "subject": subject_id,
                    "n_channels": n_channels,
                    "outer_fold": outer_fold,
                    "outer_splits": outer_splits,
                    "seed": seed,
                    **metrics,
                    "best_params": json.dumps(best_params, sort_keys=True),
                    "best_inner_balanced_accuracy": best_inner_score,
                })
                print(
                    f"Subject {subject_id:02d} | {n_channels:3d} ch | "
                    f"fold {outer_fold:02d}/{outer_splits} | "
                    f"bal={metrics['balanced_accuracy']:.3f}"
                )

    frame = pd.DataFrame(feature_results)
    frame.to_csv(
        RESULTS_DIR / "paper_comparable_subject_specific_svm.csv",
        index=False,
    )
    summary = (
        frame.groupby(["n_channels", "subject"])
        [["accuracy", "balanced_accuracy", "macro_f1"]]
        .agg(["mean", "std"])
    )
    display(summary)
    return frame, summary


# Recommended final progression (run one line at a time):
# RUN_MODE = "final"  # Then rerun the experiment-definition cells above.
# run_paper_comparable_svm(channel_counts=CHANNEL_COUNTS)
# run_loso(pipelines=("svm",))
# run_loso(pipelines=("eegnet",), deep_seeds=FINAL_SEEDS)
# run_loso(pipelines=("labram",), deep_seeds=FINAL_SEEDS)


# Paper-comparable deep-learning evaluation

This section adds subject-specific outer cross-validation for **EEGNet** and **pretrained LaBraM**. Each outer test fold remains untouched while a stratified subset of the outer-training trials is used for early stopping.

The evaluator is restart-safe:

- every completed fold has its own model bundle, predictions, metrics, and training history;
- each completed LaBraM training stage has a separate checkpoint;
- `resume=True` skips exact completed configurations after a runtime restart;
- final metrics are computed from concatenated out-of-fold predictions for each subject, then averaged across subjects and seeds.

The ordinary development-run flags are disabled in this notebook so that **Run All** does not first launch the unrelated fixed-split experiments.


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
import gc
import math


PAPER_DEEP_PROTOCOL_VERSION = "1.0"
PAPER_DEEP_RESULTS_DIR = RESULTS_DIR / "paper_comparable_deep"
PAPER_DEEP_CHECKPOINT_DIR = CHECKPOINT_DIR / "paper_comparable_deep"
PAPER_DEEP_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PAPER_DEEP_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def _paper_json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Cannot serialize {type(value).__name__}")


def _paper_signature(payload):
    encoded = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        default=_paper_json_default,
    ).encode()
    return hashlib.sha256(encoded).hexdigest()[:16]


def _atomic_json_dump(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            default=_paper_json_default,
        )
        + "\n"
    )
    temporary.replace(path)


def _atomic_csv_dump(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def _atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def _torch_load_cpu(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(path, map_location="cpu")


def prepare_paper_deep_base_data(n_channels, pipeline):
    """Prepare channel/model-specific data without fitting fold statistics."""
    pipeline = pipeline.lower()
    indices = np.asarray(channel_configs_idx[n_channels], dtype=int)
    data = np.asarray(X[:, indices, :], dtype=np.float32)

    if pipeline == "eegnet":
        return data, float(fs)

    if pipeline != "labram":
        raise ValueError("pipeline must be 'eegnet' or 'labram'")

    data = resample_array(data, float(fs), LABRAM_SFREQ)
    data = data * 1e6 / 100.0

    remainder = data.shape[-1] % LABRAM_PATCH_SIZE
    if remainder:
        padding = LABRAM_PATCH_SIZE - remainder
        left = padding // 2
        right = padding - left
        data = np.pad(
            data,
            ((0, 0), (0, 0), (left, right)),
            mode="constant",
            constant_values=0.0,
        )

    return np.asarray(data, dtype=np.float32), float(LABRAM_SFREQ)


def prepare_paper_deep_fold(
    base_data,
    train_indices,
    validation_indices,
    test_indices,
    pipeline,
):
    """Build one leakage-safe train/validation/test fold."""
    fold_indices = {
        "train": np.asarray(train_indices, dtype=int),
        "val": np.asarray(validation_indices, dtype=int),
        "test": np.asarray(test_indices, dtype=int),
    }

    if pipeline == "eegnet":
        training_data = base_data[fold_indices["train"]]
        mean = training_data.mean(
            axis=(0, 2),
            keepdims=True,
            dtype=np.float64,
        )
        std = training_data.std(
            axis=(0, 2),
            keepdims=True,
            dtype=np.float64,
        )
        std = np.maximum(std, 1e-8)
        arrays = {
            name: (
                (base_data[index] - mean) / std
            ).astype(np.float32)
            for name, index in fold_indices.items()
        }
    else:
        arrays = {
            name: np.asarray(base_data[index], dtype=np.float32)
            for name, index in fold_indices.items()
        }

    labels = {
        name: np.asarray(y[index], dtype=np.int64)
        for name, index in fold_indices.items()
    }
    return arrays, labels


def _labram_encoder_cache_path(n_channels, n_times):
    identity = {
        "version": PAPER_DEEP_PROTOCOL_VERSION,
        "repo": LABRAM_REPO_ID,
        "n_channels": int(n_channels),
        "n_times": int(n_times),
        "channel_hash": channel_config_hash(n_channels),
        "sfreq": LABRAM_SFREQ,
        "patch_size": LABRAM_PATCH_SIZE,
    }
    signature = _paper_signature(identity)
    return (
        PAPER_DEEP_CHECKPOINT_DIR
        / "labram_encoder_cache"
        / f"labram_encoder_{n_channels}ch_{n_times}t_{signature}.pt"
    )


def get_cached_labram_encoder(n_channels, n_times):
    """Load LaBraM once per geometry and cache its pretrained encoder."""
    cache_path = _labram_encoder_cache_path(n_channels, n_times)
    if cache_path.exists():
        payload = _torch_load_cpu(cache_path)
        print(f"Reusing LaBraM encoder cache: {cache_path.name}")
        return payload

    set_all_seeds(42)
    model, version, projection_shape, load_report = load_adapted_labram(
        n_channels,
        n_times=n_times,
        pretrained=True,
    )
    encoder_state = {
        name: value.detach().cpu().clone()
        for name, value in model.state_dict().items()
        if not name.startswith("final_layer")
    }
    payload = {
        "state_dict": encoder_state,
        "repo": LABRAM_REPO_ID,
        "braindecode_version": version,
        "projection_shape": projection_shape,
        "load_report": load_report,
        "n_channels": int(n_channels),
        "n_times": int(n_times),
    }
    _atomic_torch_save(payload, cache_path)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return payload


def _make_fold_labram(n_channels, n_times, encoder_payload, seed):
    """Create a fresh classification head over the cached pretrained encoder."""
    set_all_seeds(seed)
    model, current_version, projection_shape, _ = load_adapted_labram(
        n_channels,
        n_times=n_times,
        pretrained=False,
    )
    incompatible = model.load_state_dict(
        encoder_payload["state_dict"],
        strict=False,
    )
    unexpected = list(incompatible.unexpected_keys)
    invalid_missing = [
        name
        for name in incompatible.missing_keys
        if not name.startswith("final_layer")
    ]
    if unexpected or invalid_missing:
        raise RuntimeError(
            "Cached LaBraM encoder is incompatible with the current model: "
            f"missing={invalid_missing}, unexpected={unexpected}"
        )
    if tuple(projection_shape) != tuple(encoder_payload["projection_shape"]):
        raise RuntimeError("LaBraM projection changed since the cache was created.")
    return model, current_version, projection_shape


def _stage_is_complete(stage_metadata_path, checkpoint_path, signature):
    if not stage_metadata_path.exists() or not checkpoint_path.exists():
        return False
    try:
        metadata = json.loads(stage_metadata_path.read_text())
    except (OSError, json.JSONDecodeError):
        return False
    return (
        metadata.get("status") == "complete"
        and metadata.get("signature") == signature
    )


def _fit_or_resume_stage(
    model,
    loaders,
    optimizer,
    epochs,
    patience,
    scheduler,
    task_dir,
    stage,
    signature,
    description,
    resume,
):
    checkpoint_path = task_dir / f"stage_{stage}_best.pt"
    history_path = task_dir / f"stage_{stage}_history.csv"
    metadata_path = task_dir / f"stage_{stage}_metadata.json"

    if resume and _stage_is_complete(
        metadata_path,
        checkpoint_path,
        signature,
    ):
        model.load_state_dict(_torch_load_cpu(checkpoint_path))
        metadata = json.loads(metadata_path.read_text())
        history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
        print(f"Resumed completed {description}")
        return history, int(metadata["best_epoch"]), checkpoint_path

    history, best_epoch = fit_torch_stage(
        model,
        loaders,
        optimizer,
        epochs=epochs,
        patience=patience,
        checkpoint_path=checkpoint_path,
        description=description,
        scheduler=scheduler,
    )
    _atomic_csv_dump(history, history_path)
    _atomic_json_dump(
        {
            "status": "complete",
            "signature": signature,
            "stage": stage,
            "best_epoch": int(best_epoch),
            "epochs_requested": int(epochs),
            "patience": int(patience),
            "checkpoint": str(checkpoint_path),
        },
        metadata_path,
    )
    return history, best_epoch, checkpoint_path


def _task_is_complete(task_dir, signature):
    metadata_path = task_dir / "metadata.json"
    prediction_path = task_dir / "test_predictions.csv"
    bundle_path = task_dir / "final_model_bundle.pt"
    if not (metadata_path.exists() and prediction_path.exists() and bundle_path.exists()):
        return False
    try:
        metadata = json.loads(metadata_path.read_text())
    except (OSError, json.JSONDecodeError):
        return False
    return (
        metadata.get("status") == "complete"
        and metadata.get("signature") == signature
    )


def _train_paper_deep_fold(
    pipeline,
    n_channels,
    arrays,
    labels,
    fold_seed,
    task_dir,
    signature,
    resume,
    eegnet_epochs,
    eegnet_patience,
    eegnet_batch_size,
    labram_stages,
    labram_batch_size,
    labram_encoder_payload=None,
):
    set_all_seeds(fold_seed)
    augment = pipeline == "eegnet"
    batch_size = eegnet_batch_size if pipeline == "eegnet" else labram_batch_size
    loaders = make_loaders(
        arrays,
        labels,
        batch_size=batch_size,
        seed=fold_seed,
        augment_train=augment,
    )

    histories = []
    best_epochs = {}
    stage_checkpoints = {}

    if pipeline == "eegnet":
        model = EEGNet(n_channels=n_channels, n_classes=4).to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-4,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=eegnet_epochs,
            eta_min=1e-5,
        )
        history, best_epoch, checkpoint_path = _fit_or_resume_stage(
            model,
            loaders,
            optimizer,
            epochs=eegnet_epochs,
            patience=eegnet_patience,
            scheduler=scheduler,
            task_dir=task_dir,
            stage="full",
            signature=signature,
            description=(
                f"Paper EEGNet {n_channels}ch seed{fold_seed}"
            ),
            resume=resume,
        )
        histories.append(history.assign(stage="full"))
        best_epochs["full"] = int(best_epoch)
        stage_checkpoints["full"] = str(checkpoint_path)
        model_metadata = {
            "model": "EEGNet_augmented",
            "model_sfreq": float(fs),
            "augmentation": True,
        }
    else:
        model, braindecode_version, projection_shape = _make_fold_labram(
            n_channels,
            n_times=arrays["train"].shape[-1],
            encoder_payload=labram_encoder_payload,
            seed=fold_seed,
        )
        model = model.to(device)
        for stage, epochs, patience in labram_stages:
            configure_labram_stage(model, stage)
            optimizer = make_labram_optimizer(model, stage)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=epochs,
                eta_min=1e-6,
            )
            history, best_epoch, checkpoint_path = _fit_or_resume_stage(
                model,
                loaders,
                optimizer,
                epochs=epochs,
                patience=patience,
                scheduler=scheduler,
                task_dir=task_dir,
                stage=stage,
                signature=signature,
                description=(
                    f"Paper LaBraM {stage} {n_channels}ch seed{fold_seed}"
                ),
                resume=resume,
            )
            histories.append(history.assign(stage=stage))
            best_epochs[stage] = int(best_epoch)
            stage_checkpoints[stage] = str(checkpoint_path)
        model_metadata = {
            "model": "LaBraM_pretrained",
            "model_sfreq": LABRAM_SFREQ,
            "pretrained_repo": LABRAM_REPO_ID,
            "braindecode_version": braindecode_version,
            "projection_shape": tuple(projection_shape),
            "projection_trainable": False,
            "augmentation": False,
        }

    history_frame = pd.concat(histories, ignore_index=True)
    _atomic_csv_dump(history_frame, task_dir / "training_history.csv")
    return model, loaders, best_epochs, stage_checkpoints, model_metadata


def _paper_subject_metrics(oof_predictions):
    rows = []
    group_columns = ["pipeline", "n_channels", "subject", "seed"]
    for keys, frame in oof_predictions.groupby(group_columns, sort=True):
        if frame["trial_index"].duplicated().any():
            raise RuntimeError(f"Duplicate out-of-fold predictions for {keys}")
        metrics = classification_metrics(
            frame["y_true"].to_numpy(),
            frame["y_pred"].to_numpy(),
        )
        rows.append({
            **dict(zip(group_columns, keys)),
            "n_trials": len(frame),
            **metrics,
        })
    return pd.DataFrame(rows)


def run_paper_comparable_deep(
    pipeline,
    channel_counts=(128,),
    subject_ids=None,
    seeds=(42,),
    outer_folds=10,
    inner_validation_fraction=0.15,
    resume=True,
    eegnet_epochs=None,
    eegnet_patience=None,
    eegnet_batch_size=32,
    labram_stages=None,
    labram_batch_size=16,
):
    """
    Subject-specific nested evaluation for EEGNet or pretrained LaBraM.

    The outer fold is used only for evaluation. Early stopping uses a
    stratified validation subset drawn exclusively from the outer-training
    trials. Completed folds and completed LaBraM stages are resumed from disk.
    """
    pipeline = pipeline.lower()
    if pipeline not in {"eegnet", "labram"}:
        raise ValueError("pipeline must be 'eegnet' or 'labram'")
    if not 0.0 < inner_validation_fraction < 0.5:
        raise ValueError("inner_validation_fraction must be between 0 and 0.5")

    channel_counts = tuple(int(value) for value in channel_counts)
    unknown_channels = set(channel_counts) - set(CHANNEL_COUNTS)
    if unknown_channels:
        raise ValueError(f"Unknown channel counts: {sorted(unknown_channels)}")

    subject_ids = (
        sorted(np.unique(subjects).tolist())
        if subject_ids is None
        else [int(value) for value in subject_ids]
    )
    seeds = tuple(int(value) for value in seeds)
    eegnet_epochs = eegnet_epochs or (60 if RUN_MODE == "quick" else 120)
    eegnet_patience = eegnet_patience or (12 if RUN_MODE == "quick" else 20)
    labram_stages = labram_stages or (
        ("head", 10, 6),
        ("partial", 25, 10),
        ("full", 40, 12),
    )
    labram_stages = tuple(
        (str(stage), int(epochs), int(patience))
        for stage, epochs, patience in labram_stages
    )

    run_config = {
        "protocol_version": PAPER_DEEP_PROTOCOL_VERSION,
        "pipeline": pipeline,
        "channel_counts": channel_counts,
        "subject_ids": subject_ids,
        "seeds": seeds,
        "outer_folds": int(outer_folds),
        "inner_validation_fraction": float(inner_validation_fraction),
        "eegnet_epochs": int(eegnet_epochs),
        "eegnet_patience": int(eegnet_patience),
        "eegnet_batch_size": int(eegnet_batch_size),
        "labram_stages": labram_stages,
        "labram_batch_size": int(labram_batch_size),
        "input_shape": tuple(X.shape),
        "input_sfreq": float(fs),
    }
    run_signature = _paper_signature(run_config)
    run_results_dir = PAPER_DEEP_RESULTS_DIR / (
        f"{pipeline}_{run_signature}"
    )
    run_results_dir.mkdir(parents=True, exist_ok=True)
    _atomic_json_dump(run_config, run_results_dir / "run_config.json")

    prediction_frames = []
    fold_rows = []

    for n_channels in channel_counts:
        print(f"\nPreparing {pipeline} data for {n_channels} channels...")
        base_data, model_sfreq = prepare_paper_deep_base_data(
            n_channels,
            pipeline,
        )
        encoder_payload = None
        if pipeline == "labram":
            encoder_payload = get_cached_labram_encoder(
                n_channels,
                n_times=base_data.shape[-1],
            )

        for seed in seeds:
            for subject_id in subject_ids:
                subject_indices = np.flatnonzero(subjects == subject_id)
                if not len(subject_indices):
                    raise ValueError(f"Subject {subject_id} is not present")
                subject_labels = y[subject_indices]
                class_counts = np.bincount(subject_labels)
                minimum_class_count = int(class_counts[class_counts > 0].min())
                actual_outer_folds = min(int(outer_folds), minimum_class_count)
                if actual_outer_folds < 2:
                    raise ValueError(
                        f"Subject {subject_id} has insufficient class counts"
                    )

                outer_cv = StratifiedKFold(
                    n_splits=actual_outer_folds,
                    shuffle=True,
                    random_state=seed,
                )

                for outer_fold, (outer_train_local, test_local) in enumerate(
                    outer_cv.split(subject_indices, subject_labels),
                    start=1,
                ):
                    outer_train_indices = subject_indices[outer_train_local]
                    test_indices = subject_indices[test_local]
                    fold_seed = int(
                        seed * 100_000
                        + int(subject_id) * 100
                        + outer_fold
                    )

                    outer_training_labels = y[outer_train_indices]
                    inner_class_count = len(np.unique(outer_training_labels))
                    validation_size = max(
                        inner_class_count,
                        int(math.ceil(
                            inner_validation_fraction
                            * len(outer_train_indices)
                        )),
                    )
                    validation_size = min(
                        validation_size,
                        len(outer_train_indices) - inner_class_count,
                    )
                    if validation_size < inner_class_count:
                        raise ValueError(
                            f"Subject {subject_id} fold {outer_fold} has too "
                            "few training trials for stratified early stopping"
                        )

                    inner_splitter = StratifiedShuffleSplit(
                        n_splits=1,
                        test_size=validation_size,
                        random_state=fold_seed,
                    )
                    train_local, validation_local = next(
                        inner_splitter.split(
                            outer_train_indices,
                            outer_training_labels,
                        )
                    )
                    train_indices = outer_train_indices[train_local]
                    validation_indices = outer_train_indices[validation_local]

                    task_config = {
                        **run_config,
                        "channel_counts": (n_channels,),
                        "subject_ids": (subject_id,),
                        "seeds": (seed,),
                        "actual_outer_folds": actual_outer_folds,
                        "outer_fold": outer_fold,
                        "fold_seed": fold_seed,
                        "channel_hash": channel_config_hash(n_channels),
                        "model_sfreq": model_sfreq,
                        "train_index_hash": hashlib.sha256(
                            np.asarray(train_indices, dtype=np.int64).tobytes()
                        ).hexdigest()[:16],
                        "validation_index_hash": hashlib.sha256(
                            np.asarray(validation_indices, dtype=np.int64).tobytes()
                        ).hexdigest()[:16],
                        "test_index_hash": hashlib.sha256(
                            np.asarray(test_indices, dtype=np.int64).tobytes()
                        ).hexdigest()[:16],
                    }
                    task_signature = _paper_signature(task_config)
                    task_dir = (
                        PAPER_DEEP_CHECKPOINT_DIR
                        / pipeline
                        / f"{n_channels}ch"
                        / f"subject_{int(subject_id):02d}"
                        / f"seed_{seed}"
                        / f"fold_{outer_fold:02d}_{task_signature}"
                    )
                    task_dir.mkdir(parents=True, exist_ok=True)

                    if resume and _task_is_complete(task_dir, task_signature):
                        predictions = pd.read_csv(
                            task_dir / "test_predictions.csv"
                        )
                        metadata = json.loads(
                            (task_dir / "metadata.json").read_text()
                        )
                        prediction_frames.append(predictions)
                        fold_rows.append(metadata["fold_result"])
                        print(
                            f"Resumed {pipeline} | {n_channels:3d} ch | "
                            f"subject {subject_id:02d} | "
                            f"fold {outer_fold:02d}/{actual_outer_folds}"
                        )
                        continue

                    arrays, fold_labels = prepare_paper_deep_fold(
                        base_data,
                        train_indices,
                        validation_indices,
                        test_indices,
                        pipeline,
                    )
                    model, loaders, best_epochs, stage_checkpoints, model_metadata = (
                        _train_paper_deep_fold(
                            pipeline=pipeline,
                            n_channels=n_channels,
                            arrays=arrays,
                            labels=fold_labels,
                            fold_seed=fold_seed,
                            task_dir=task_dir,
                            signature=task_signature,
                            resume=resume,
                            eegnet_epochs=eegnet_epochs,
                            eegnet_patience=eegnet_patience,
                            eegnet_batch_size=eegnet_batch_size,
                            labram_stages=labram_stages,
                            labram_batch_size=labram_batch_size,
                            labram_encoder_payload=encoder_payload,
                        )
                    )

                    criterion = nn.CrossEntropyLoss()
                    _, validation_y, validation_pred = predict_torch(
                        model,
                        loaders["val"],
                        criterion,
                    )
                    _, test_y, test_pred = predict_torch(
                        model,
                        loaders["test"],
                        criterion,
                    )
                    validation_metrics = classification_metrics(
                        validation_y,
                        validation_pred,
                    )
                    test_metrics = classification_metrics(test_y, test_pred)

                    predictions = pd.DataFrame({
                        "protocol": "paper_subject_specific",
                        "pipeline": pipeline,
                        "model": model_metadata["model"],
                        "n_channels": n_channels,
                        "subject": int(subject_id),
                        "seed": seed,
                        "outer_fold": outer_fold,
                        "trial_index": test_indices,
                        "session": sessions[test_indices],
                        "y_true": test_y,
                        "y_pred": test_pred,
                    })
                    _atomic_csv_dump(
                        predictions,
                        task_dir / "test_predictions.csv",
                    )

                    fold_result = {
                        "protocol": "paper_subject_specific",
                        "pipeline": pipeline,
                        "model": model_metadata["model"],
                        "n_channels": n_channels,
                        "subject": int(subject_id),
                        "seed": seed,
                        "outer_fold": outer_fold,
                        "outer_folds": actual_outer_folds,
                        "n_train": len(train_indices),
                        "n_validation": len(validation_indices),
                        "n_test": len(test_indices),
                        "validation_accuracy": validation_metrics["accuracy"],
                        "validation_balanced_accuracy": validation_metrics[
                            "balanced_accuracy"
                        ],
                        "validation_macro_f1": validation_metrics["macro_f1"],
                        "test_accuracy": test_metrics["accuracy"],
                        "test_balanced_accuracy": test_metrics[
                            "balanced_accuracy"
                        ],
                        "test_macro_f1": test_metrics["macro_f1"],
                        "best_epochs": json.dumps(best_epochs, sort_keys=True),
                        "task_signature": task_signature,
                        "task_directory": str(task_dir),
                    }

                    bundle_path = task_dir / "final_model_bundle.pt"
                    _atomic_torch_save(
                        {
                            "state_dict": {
                                name: value.detach().cpu().clone()
                                for name, value in model.state_dict().items()
                            },
                            "task_config": task_config,
                            "model_metadata": model_metadata,
                            "best_epochs": best_epochs,
                            "stage_checkpoints": stage_checkpoints,
                            "class_names": CLASS_NAMES,
                        },
                        bundle_path,
                    )
                    _atomic_json_dump(
                        {
                            "status": "complete",
                            "signature": task_signature,
                            "fold_result": fold_result,
                            "model_metadata": model_metadata,
                            "best_epochs": best_epochs,
                            "stage_checkpoints": stage_checkpoints,
                            "final_model_bundle": str(bundle_path),
                        },
                        task_dir / "metadata.json",
                    )

                    prediction_frames.append(predictions)
                    fold_rows.append(fold_result)
                    print(
                        f"{pipeline} | {n_channels:3d} ch | "
                        f"subject {subject_id:02d} | "
                        f"fold {outer_fold:02d}/{actual_outer_folds} | "
                        f"test bal={test_metrics['balanced_accuracy']:.3f}"
                    )

                    del model, loaders, arrays, fold_labels
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

        del base_data
        gc.collect()

    if not prediction_frames:
        raise RuntimeError("No paper-comparable deep-learning folds were produced")

    oof_predictions = pd.concat(prediction_frames, ignore_index=True)
    fold_metrics = pd.DataFrame(fold_rows)
    subject_metrics = _paper_subject_metrics(oof_predictions)
    summary = (
        subject_metrics
        .groupby(["pipeline", "n_channels"])[
            ["accuracy", "balanced_accuracy", "macro_f1"]
        ]
        .agg(["mean", "std"])
    )

    _atomic_csv_dump(
        oof_predictions,
        run_results_dir / "oof_predictions.csv",
    )
    _atomic_csv_dump(
        fold_metrics,
        run_results_dir / "fold_metrics.csv",
    )
    _atomic_csv_dump(
        subject_metrics,
        run_results_dir / "subject_metrics.csv",
    )
    summary.to_csv(run_results_dir / "summary.csv")

    print(f"\nSaved paper-comparable results to: {run_results_dir}")
    display(summary)
    return subject_metrics, summary


## Easy runner

Start with EEGNet at 128 channels and one seed. Change only its switch to `True`. After that run completes, turn it back off and enable LaBraM. Both calls use subject-specific 10-fold evaluation and resume completed folds automatically.

For a short functional test, set `PAPER_DEEP_SUBJECT_IDS = (1,)`, `PAPER_DEEP_CHANNEL_COUNTS = (16,)`, and `PAPER_DEEP_OUTER_FOLDS = 2` before enabling a model.


In [ ]:
RUN_PAPER_EEGNET = False
RUN_PAPER_LABRAM = False

PAPER_DEEP_CHANNEL_COUNTS = (128,)
PAPER_DEEP_SUBJECT_IDS = None  # None means all subjects.
PAPER_DEEP_SEEDS = (42,)
PAPER_DEEP_OUTER_FOLDS = 10

paper_eegnet_frame = None
paper_eegnet_summary = None
paper_labram_frame = None
paper_labram_summary = None

if RUN_PAPER_EEGNET:
    paper_eegnet_frame, paper_eegnet_summary = (
        run_paper_comparable_deep(
            pipeline="eegnet",
            channel_counts=PAPER_DEEP_CHANNEL_COUNTS,
            subject_ids=PAPER_DEEP_SUBJECT_IDS,
            seeds=PAPER_DEEP_SEEDS,
            outer_folds=PAPER_DEEP_OUTER_FOLDS,
            resume=True,
        )
    )

if RUN_PAPER_LABRAM:
    paper_labram_frame, paper_labram_summary = (
        run_paper_comparable_deep(
            pipeline="labram",
            channel_counts=PAPER_DEEP_CHANNEL_COUNTS,
            subject_ids=PAPER_DEEP_SUBJECT_IDS,
            seeds=PAPER_DEEP_SEEDS,
            outer_folds=PAPER_DEEP_OUTER_FOLDS,
            resume=True,
        )
    )

if not RUN_PAPER_EEGNET and not RUN_PAPER_LABRAM:
    print(
        "Paper-comparable deep evaluation is ready. "
        "Enable one RUN_PAPER_* switch above."
    )


## Recommended progression

1. Run the short functional test for EEGNet.
2. Use 128 channels, all subjects, one seed for EEGNet.
3. Use 128 channels, all subjects, one seed for LaBraM.
4. Only after those complete, change `PAPER_DEEP_CHANNEL_COUNTS` to `(128, 64, 32, 16)`.
5. For the final repeated-seed analysis, change `PAPER_DEEP_SEEDS` to `FINAL_SEEDS`.

Do not delete `checkpoints_recommended/paper_comparable_deep`; it is the resume state. Aggregate outputs are written under `results_recommended/paper_comparable_deep`.
